Question 2: Customer Purchase Analysis

Using the provided dataset 'customer_purchases.csv', perform the following analyses:

1. Calculate the following metrics per customer:
   - Total amount spent
   - Average purchase value
   - Number of purchases
   - Most frequently bought category

2. Create a summary DataFrame with:
   - Top 5 customers by total spend
   - Bottom 5 customers by total spend

3. Calculate the monthly purchase trends:
   - Total sales per month
   - Average purchase value per month

Bonus: Identify any customers who haven't made a purchase in the last 3 months

In [1]:
import numpy as np
import pandas as pd

from datetime import date, timedelta

In [2]:
# First, let's load and take a look at the dataset
purchases_df = pd.read_csv('../data/raw/customer_purchases.csv')
purchases_df.sample(20)

,purchase_id,customer_id,purchase_date,category,amount
583,400,CUST_023,2023-06-21,Books,106.05
792,95,CUST_040,2023-08-25,Home & Garden,117.76
181,796,CUST_026,2023-02-19,Home & Garden,57.49
191,253,CUST_097,2023-02-24,Clothing,116.82
946,648,CUST_039,2023-10-15,Clothing,74.84
188,922,CUST_021,2023-02-22,Sports,137.30
606,255,CUST_059,2023-06-27,Home & Garden,109.30
146,451,CUST_070,2023-02-08,Books,78.34
354,799,CUST_086,2023-04-15,Clothing,68.62
611,879,CUST_079,2023-06-28,Sports,123.46


In [3]:
# get count and mean. Unfortunately .describe() does not perform sums anymore
metrics_base = purchases_df.groupby('customer_id')[['amount'][0]].describe().rename(
    columns = {
        'count': 'total_purchases',
        'mean': 'average_amount',
    }
)[['total_purchases', 'average_amount']]
# get sum()
sum_base = purchases_df.groupby('customer_id')[['amount']].sum().rename(columns={'amount': 'total_spent'})

# combine the two dfs
metrics_df = metrics_base.merge(sum_base, on='customer_id', how='inner')

# we still lack most common category, but this is what we have at this point
metrics_df

,total_purchases,average_amount,total_spent
customer_id,,,
CUST_001,16.0,96.921875,1550.75
CUST_002,13.0,101.057692,1313.75
CUST_003,12.0,110.326667,1323.92
CUST_004,11.0,100.244545,1102.69
CUST_005,11.0,109.024545,1199.27
...,...,...,...
CUST_096,13.0,106.339231,1382.41
CUST_097,9.0,106.470000,958.23
CUST_098,7.0,92.350000,646.45


In [4]:
# Here we go. A dataframe with total spendings, average amount spend, total purchases and most common category 
# bought from, all grouped by customer

metrics_df['preferred_category'] = purchases_df.groupby(['customer_id'])['category'].agg(pd.Series.mode)
metrics_df[['total_spent', 'average_amount', 'total_purchases', 'preferred_category']]

,total_spent,average_amount,total_purchases,preferred_category
customer_id,,,,
CUST_001,1550.75,96.921875,16.0,Home & Garden
CUST_002,1313.75,101.057692,13.0,Electronics
CUST_003,1323.92,110.326667,12.0,"[Clothing, Electronics]"
CUST_004,1102.69,100.244545,11.0,"[Electronics, Sports]"
CUST_005,1199.27,109.024545,11.0,Electronics
...,...,...,...,...
CUST_096,1382.41,106.339231,13.0,Home & Garden
CUST_097,958.23,106.470000,9.0,Sports
CUST_098,646.45,92.350000,7.0,Home & Garden


### Top 5 and bottom 5 customers per total spent

In [5]:
total_spent_ranking = metrics_df.sort_values('total_spent', ascending=False)[['total_spent']]

top_5 = total_spent_ranking[:5]
bottom_5 = total_spent_ranking[-5:]

In [6]:
top_5

,total_spent
customer_id,
CUST_062,2102.22
CUST_090,2040.24
CUST_099,2000.90
CUST_092,1779.33
CUST_033,1775.29


In [7]:
bottom_5

,total_spent
customer_id,
CUST_091,480.91
CUST_040,459.16
CUST_083,450.57
CUST_011,373.26
CUST_031,356.43


### Monthly metrics

In [8]:
# to calculate metrics based on purchase month, we need first to get the month start for every purchase

purchases_df['purchase_date_month_start'] = purchases_df['purchase_date'].apply(lambda x: f"{x[:-3]}-01")

monthly_metrics = purchases_df.groupby('purchase_date_month_start')[['amount'][0]].describe()[['mean']].rename(
    columns={'mean': 'average'}
)
monthly_metrics['total'] = purchases_df.groupby('purchase_date_month_start')[['amount'][0]].sum()

In [9]:
monthly_metrics

,average,total
purchase_date_month_start,,
2023-01-01,102.720328,12531.88
2023-02-01,104.238824,8860.30
2023-03-01,101.432079,10244.64
2023-04-01,100.293267,10129.62
2023-05-01,101.720102,9968.57
2023-06-01,103.816018,11731.21
2023-07-01,101.929250,8154.34
2023-08-01,102.948850,11633.22
2023-09-01,100.950814,8681.77


### Detractors

In [10]:
# to make comparisons easier, let us convert the date from string to python date format 
# if we saved purchase_date into itself, this cell would raise an error if ran twice in a row
# I dislike that, so let's save the resulting date objects into a separate column
purchases_df['purchase_date_date'] = purchases_df['purchase_date'].apply(lambda x: date(*[int(d) for d in x.split('-')]))

last_purchase_df = purchases_df.groupby('customer_id')[['purchase_date_date']].max().rename(columns={
    'purchase_date_date': 'last_purchase'
})

In [11]:
last_purchase_df[last_purchase_df['last_purchase'] < date.today() - timedelta(days=90)]

,last_purchase
customer_id,
CUST_001,2023-10-22
CUST_002,2023-10-13
CUST_003,2023-10-02
CUST_004,2023-10-23
CUST_005,2023-08-19
...,...
CUST_096,2023-10-20
CUST_097,2023-10-19
CUST_098,2023-10-22


Well, apparently, no customer has made a purchase in the last 90 days. Time to renew this dataset (or renew the customer base). 